In [89]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [81]:
np.random.seed(42)

# 1. GENERATE RANDOM CLASSIFICATION DATA
# Create a dataset with 1000 samples, 10 features, and 2 classes (binary)
X_raw, y_raw = make_classification(
    n_samples=100, n_features=10, n_classes=2, random_state=42
)

# Split into 80% train and 20% test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Scale features for better neural network convergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Now convert these sets to tensors, ensuring float64 for consistency
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)

y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [82]:
# Implementation of a Neural Network
class ANN(nn.Module):
  def __init__(self,input_dim):
    super().__init__()
    self.ann = nn.Sequential(
        nn.Linear(input_dim,2), # Pass to three hidden layer neurons
        nn.ReLU(),# Apply the ReLu function in hidden layers
        nn.Linear(2,1), # Pass the hidden layer outputs to output to output layer
        nn.Sigmoid() # Pass the ouput weighted sum to sigmoid function for classifying into 0 or 1
    )

  def forward(self,X_train):
    output = self.ann(X_train)
    return output

In [83]:
model = ANN(X_train.shape[1])

In [84]:
model.forward(X_train)

tensor([[0.4279],
        [0.4285],
        [0.4285],
        [0.4285],
        [0.4282],
        [0.4285],
        [0.4277],
        [0.4331],
        [0.4285],
        [0.4273],
        [0.4285],
        [0.4287],
        [0.4410],
        [0.4384],
        [0.4279],
        [0.4346],
        [0.4285],
        [0.4282],
        [0.4279],
        [0.4283],
        [0.4298],
        [0.4291],
        [0.4285],
        [0.4285],
        [0.4311],
        [0.4301],
        [0.4285],
        [0.4301],
        [0.4285],
        [0.4315],
        [0.4285],
        [0.4285],
        [0.4285],
        [0.4282],
        [0.4324],
        [0.4280],
        [0.4285],
        [0.4319],
        [0.4280],
        [0.4281],
        [0.4285],
        [0.4285],
        [0.4277],
        [0.4290],
        [0.4387],
        [0.4285],
        [0.4281],
        [0.4345],
        [0.4327],
        [0.4285],
        [0.4285],
        [0.4275],
        [0.4285],
        [0.4285],
        [0.4285],
        [0

In [88]:
!pip install torchinfo
from torchinfo import summary
summary(model, input_size=(X_train.shape[1], 5))

RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: []

In [90]:
summary(model, input_size=(1, X_train.shape[1])) # Correct input_size for a single sample

Layer (type:depth-idx)                   Output Shape              Param #
ANN                                      [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 2]                    22
│    └─ReLU: 2-2                         [1, 2]                    --
│    └─Linear: 2-3                       [1, 1]                    3
│    └─Sigmoid: 2-4                      [1, 1]                    --
Total params: 25
Trainable params: 25
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Accessing Weights and Biases

YouSince your `ANN` model is built using `nn.Sequential`, you can access individual layers by their index in the sequence. Each `nn.Linear` layer has `weight` and `bias` attributes.

- The first linear layer is `model.ann[0]`.
- The second linear layer is `model.ann[2]`.

In [91]:
print("Weights of the first linear layer:\n", model.ann[0].weight)
print("Bias of the first linear layer:\n", model.ann[0].bias)

print("\nWeights of the second linear layer:\n", model.ann[2].weight)
print("Bias of the second linear layer:\n", model.ann[2].bias)

Weights of the first linear layer:
 Parameter containing:
tensor([[ 0.2445,  0.1313,  0.2750,  0.0626, -0.0934,  0.3132,  0.1538,  0.0830,
         -0.0171,  0.2638],
        [-0.2113,  0.1910,  0.1558, -0.1740,  0.0786,  0.1159,  0.1746,  0.2461,
         -0.1031, -0.2760]], requires_grad=True)
Bias of the first linear layer:
 Parameter containing:
tensor([-0.1842, -0.0457], requires_grad=True)

Weights of the second linear layer:
 Parameter containing:
tensor([[ 0.0481, -0.0042]], requires_grad=True)
Bias of the second linear layer:
 Parameter containing:
tensor([-0.2879], requires_grad=True)


### Training the ANN Model

To train the model, we need to define a **loss function** and an **optimizer**. For binary classification with a sigmoid output, `nn.BCELoss` is a suitable choice. For the optimizer, `torch.optim.Adam` is a good general-purpose option.

Here's how to set up and run a basic training loop:

In [92]:
# Define hyperparameters
learning_rate = 0.01
epochs = 100

# Define Loss Function and Optimizer
criterion = nn.BCELoss() # Binary Cross Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(epochs):
    # Zero gradients
    optimizer.zero_grad()

    # Forward pass
    outputs = model(X_train)

    # Calculate loss
    # Ensure y_train is float and same shape as outputs for BCELoss
    loss = criterion(outputs, y_train.float().unsqueeze(1))

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6734
Epoch [20/100], Loss: 0.6219
Epoch [30/100], Loss: 0.5504
Epoch [40/100], Loss: 0.4749
Epoch [50/100], Loss: 0.4029
Epoch [60/100], Loss: 0.3347
Epoch [70/100], Loss: 0.2820
Epoch [80/100], Loss: 0.2448
Epoch [90/100], Loss: 0.2173
Epoch [100/100], Loss: 0.1960
